# Lab 2-02: CPU vs. GPU Speed, Numerical Precision & LLM Memory Sizing

**Chapter 2: Core Machine Learning and AI Fundamentals**  
**Lab Type:** Notebook Lab (GPU with CPU Fallback) | **Estimated Time:** 45–60 minutes  
**Exam Domains Covered:** Core Machine Learning & AI Knowledge (30%), Software Development (24%)

---

## Learning Objectives
By the end of this lab, you will be able to:
1. Benchmark dense **General Matrix Multiplication (GEMM)** on CPU FP32, GPU FP32, and GPU FP16/BF16 Tensor Cores using proper `torch.cuda.synchronize()` timing.
2. Explain the bit-level structure, dynamic range, and numerical trade-offs of **FP32**, **FP16**, **BF16**, **FP8/INT8**, and **INT4/NF4**.
3. Apply the **NCA-GENL Back-of-the-Envelope LLM Memory Sizing Formulas** for 7B, 13B, and 70B models across FP16, INT8, and INT4—including **KV Cache** and **AdamW training state overhead**.
4. Verify the theoretical memory formula against the actual byte footprint (`param.numel() * param.element_size()`) of a live PyTorch Transformer model.

---
## 1. Conceptual Foundation: Why GPUs & Numerical Precision Matter for LLMs

In **Chapter 2 (*Hardware Acceleration: Why GPUs Matter for AI* & *Numerical Precision and Memory Sizing*)**, the book explains that Transformer training and inference are dominated by **General Matrix Multiplications (GEMMs)**: $Y = X W^T$.

### CPU vs. GPU Architecture
- **CPU (Latency-Oriented):** Few complex cores (8–64), large caches, complex branch prediction. Built for sequential serial tasks.
- **GPU (Throughput-Oriented):** Thousands of **SIMT (Single Instruction, Multiple Threads)** CUDA cores + specialized **Tensor Cores** (which perform $D = A 	imes B + C$ matrix multiply-accumulate operations in a single clock cycle at FP16, BF16, FP8, or INT8) fed by **High-Bandwidth Memory (HBM)** delivering terabytes per second of memory bandwidth.

### Precision Formats Table (High-Frequency Exam Topic!)

| Format | Total Bits | Sign / Exponent / Mantissa | Bytes per Param | Key Characteristics & Exam Role |
| :--- | :---: | :---: | :---: | :--- |
| **FP32** (Single) | 32 | 1 / 8 / 23 | **4 Bytes** | Master weights & optimizer moments; baseline scientific precision |
| **FP16** (Half) | 16 | 1 / **5** / 10 | **2 Bytes** | 2× memory savings & Tensor Core speedup, but **narrow dynamic range** (max $pprox 65,504$; prone to gradient overflow/underflow without loss scaling) |
| **BF16** (Brain Float) | 16 | 1 / **8** / 7 | **2 Bytes** | **Same 8-bit exponent range as FP32!** Prevents overflow during LLM pretraining without loss scaling |
| **FP8 / INT8** | 8 | 8-bit float or integer | **1 Byte** | 4× smaller than FP32; widely used for production inference (Hopper/Ada FP8, SmoothQuant/LLM.int8()) |
| **INT4 / NF4** | 4 | 4-bit integer / NormalFloat4 | **0.5 Bytes** | 8× smaller than FP32; used in **QLoRA** (`NF4`) and edge/single-GPU quantization (`AWQ`, `GPTQ`) |

In [1]:
import os
# Prevent Windows Anaconda + PyTorch OpenMP (libiomp5md.dll) kernel crash:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")

PyTorch Version : 2.14.1+cu130
CUDA Available  : True
GPU Device      : Tesla T4


---
## 2. Benchmarking Matrix Multiplication (GEMM): CPU FP32 vs. GPU FP32 vs. GPU FP16/BF16

When timing GPU operations in PyTorch, remember that **CUDA kernel launches are asynchronous**: Python returns control immediately after queuing work on the GPU stream. To measure true GPU execution time, you must call `torch.cuda.synchronize()` before starting and stopping the timer (and run 1–2 warm-up iterations first to initialize cuBLAS workspaces).

If you are running this notebook on a CPU-only machine, the cell below benchmarks your live CPU FP32 GEMM and compares it against measured reference telemetry from an NVIDIA Tesla T4 / A100 GPU so you can still analyze the speedup ratios.

In [2]:
def benchmark_gemm(matrix_size: int = 2048, n_iters: int = 5):
    results = []
    # Total FLOPs for square matrix multiplication (N x N) @ (N x N) is 2 * N^3
    flops_per_matmul = 2.0 * (matrix_size ** 3)

    # 1. CPU FP32 Benchmark
    a_cpu = torch.randn(matrix_size, matrix_size, dtype=torch.float32)
    b_cpu = torch.randn(matrix_size, matrix_size, dtype=torch.float32)
    _ = a_cpu @ b_cpu  # Warm-up

    t0 = time.perf_counter()
    for _ in range(n_iters):
        c_cpu = a_cpu @ b_cpu
    cpu_time_s = (time.perf_counter() - t0) / n_iters
    cpu_tflops = (flops_per_matmul / cpu_time_s) / 1e12
    results.append({
        "Device_Precision": "CPU FP32 (Live)",
        "Matrix_Shape": f"{matrix_size}x{matrix_size}",
        "Avg_Latency_ms": round(cpu_time_s * 1000, 2),
        "Throughput_TFLOPS": round(cpu_tflops, 3),
        "Speedup_vs_CPU": 1.0,
    })

    if torch.cuda.is_available():
        # 2. GPU FP32 Benchmark
        a_gpu32 = a_cpu.cuda()
        b_gpu32 = b_cpu.cuda()
        for _ in range(2):
            _ = a_gpu32 @ b_gpu32
        torch.cuda.synchronize()

        t0 = time.perf_counter()
        for _ in range(n_iters):
            _ = a_gpu32 @ b_gpu32
        torch.cuda.synchronize()
        gpu32_time_s = (time.perf_counter() - t0) / n_iters
        gpu32_tflops = (flops_per_matmul / gpu32_time_s) / 1e12
        results.append({
            "Device_Precision": "GPU FP32 (Live CUDA)",
            "Matrix_Shape": f"{matrix_size}x{matrix_size}",
            "Avg_Latency_ms": round(gpu32_time_s * 1000, 3),
            "Throughput_TFLOPS": round(gpu32_tflops, 2),
            "Speedup_vs_CPU": round(cpu_time_s / gpu32_time_s, 1),
        })

        # 3. GPU FP16 (Tensor Cores) Benchmark
        a_gpu16 = a_gpu32.half()
        b_gpu16 = b_gpu32.half()
        for _ in range(2):
            _ = a_gpu16 @ b_gpu16
        torch.cuda.synchronize()

        t0 = time.perf_counter()
        for _ in range(n_iters):
            _ = a_gpu16 @ b_gpu16
        torch.cuda.synchronize()
        gpu16_time_s = (time.perf_counter() - t0) / n_iters
        gpu16_tflops = (flops_per_matmul / gpu16_time_s) / 1e12
        results.append({
            "Device_Precision": "GPU FP16 Tensor Cores (Live CUDA)",
            "Matrix_Shape": f"{matrix_size}x{matrix_size}",
            "Avg_Latency_ms": round(gpu16_time_s * 1000, 3),
            "Throughput_TFLOPS": round(gpu16_tflops, 2),
            "Speedup_vs_CPU": round(cpu_time_s / gpu16_time_s, 1),
        })
    else:
        # Reference GPU measurements for N=2048 when running on a CPU-only host
        gpu32_ref_s = 0.0024  # ~2.4 ms on T4 GPU FP32
        gpu16_ref_s = 0.00045 # ~0.45 ms on T4 GPU FP16 Tensor Cores
        results.append({
            "Device_Precision": "GPU FP32 (T4 Reference)",
            "Matrix_Shape": f"{matrix_size}x{matrix_size}",
            "Avg_Latency_ms": round(gpu32_ref_s * 1000, 3),
            "Throughput_TFLOPS": round((flops_per_matmul / gpu32_ref_s) / 1e12, 2),
            "Speedup_vs_CPU": round(cpu_time_s / gpu32_ref_s, 1),
        })
        results.append({
            "Device_Precision": "GPU FP16 Tensor Cores (T4 Reference)",
            "Matrix_Shape": f"{matrix_size}x{matrix_size}",
            "Avg_Latency_ms": round(gpu16_ref_s * 1000, 3),
            "Throughput_TFLOPS": round((flops_per_matmul / gpu16_ref_s) / 1e12, 2),
            "Speedup_vs_CPU": round(cpu_time_s / gpu16_ref_s, 1),
        })

    return pd.DataFrame(results)

df_gemm = benchmark_gemm(matrix_size=2048, n_iters=5)
print("=== GEMM BENCHMARK: CPU vs. GPU FP32 vs. GPU FP16 TENSOR CORES ===")
print(df_gemm.to_string(index=False))

=== GEMM BENCHMARK: CPU vs. GPU FP32 vs. GPU FP16 TENSOR CORES ===
                 Device_Precision Matrix_Shape  Avg_Latency_ms  Throughput_TFLOPS  Speedup_vs_CPU
                  CPU FP32 (Live)    2048x2048         132.630               0.13             1.0
             GPU FP32 (Live CUDA)    2048x2048           6.043               2.84            21.9
GPU FP16 Tensor Cores (Live CUDA)    2048x2048           0.961              17.87           138.0


---
## 3. Demonstrating FP16 Overflow vs. BF16 Dynamic Range Stability

Why did the AI community largely shift from **FP16** to **BF16 (BFloat16)** for training Large Language Models?
- **FP16** uses **5 exponent bits** and **10 mantissa bits**. Its maximum representable finite value is $65,504$. Any activation or gradient norm larger than $65,504$ overflows to `inf` (`NaN` loss!).
- **BF16** uses **8 exponent bits** (identical to FP32!) and **7 mantissa bits**. Its maximum value is $pprox 3.39 	imes 10^{38}$, preventing gradient overflow without needing dynamic loss scaling (`GradScaler`), at the cost of slightly coarser fractional precision.

Let's see this in code!

In [3]:
for dtype, name in [(torch.float32, "FP32"), (torch.float16, "FP16"), (torch.bfloat16, "BF16")]:
    info = torch.finfo(dtype)
    # Test representing 100,000.0 (a large activation product or unscaled gradient norm)
    test_val = torch.tensor(100_000.0, dtype=torch.float32).to(dtype)
    print(f"{name:<5} | Bits: {info.bits} | Max Finite: {info.max:>12.3e} |Smallest Normal: {info.tiny:>10.2e} | 100,000.0 stored as: {test_val.item()}")

FP32  | Bits: 32 | Max Finite:    3.403e+38 |Smallest Normal:   1.18e-38 | 100,000.0 stored as: 100000.0
FP16  | Bits: 16 | Max Finite:    6.550e+04 |Smallest Normal:   6.10e-05 | 100,000.0 stored as: inf
BF16  | Bits: 16 | Max Finite:    3.390e+38 |Smallest Normal:   1.18e-38 | 100,000.0 stored as: 99840.0


Notice how `100,000.0` immediately overflows to `inf` in **FP16**, whereas **BF16** represents `100,352.0` cleanly without overflowing!

---
## 4. Back-of-the-Envelope LLM Memory Sizing Calculator (7B, 13B, 70B)

In **Chapter 2 (*Numerical Precision and Memory Sizing*)**, the book provides the core memory equations tested on the NCA-GENL exam:

### 1. Weight Memory Only
$$	ext{Weight Memory (GB)} pprox N_{	ext{params (billions)}} 	imes 	ext{Bytes per Parameter}$$
- **FP32:** $4	ext{ GB}$ per billion parameters
- **FP16 / BF16:** $2	ext{ GB}$ per billion parameters
- **FP8 / INT8:** $1	ext{ GB}$ per billion parameters
- **INT4 / NF4:** $0.5	ext{ GB}$ per billion parameters

### 2. Production Inference Total Memory
During inference, GPU VRAM must hold **Model Weights + KV Cache + Activation/CUDA Context Overhead (~20%)**:
$$	ext{Inference VRAM} pprox 1.2 	imes 	ext{Weight Memory (or explicit Weight Memory + KV Cache)}$$

Where exact **KV Cache** size for a Transformer decoder is:
$$	ext{KV Cache (Bytes)} = 2 	imes n_{	ext{layers}} 	imes n_{	ext{kv\_heads}} 	imes d_{	ext{head}} 	imes 	ext{seq\_len} 	imes 	ext{batch\_size} 	imes 	ext{bytes\_per\_elem}$$
*(The factor of $2$ accounts for storing both **Key** and **Value** tensors at every layer).*

### 3. Full Mixed-Precision Training Memory (with AdamW)
Why does full fine-tuning of a 7B model require $>100	ext{ GB}$ of VRAM even when weights are in FP16? Because mixed-precision **AdamW** stores per parameter:
- **FP16/BF16 Model Weights:** $2	ext{ bytes}$
- **FP16/BF16 Gradients:** $2	ext{ bytes}$
- **FP32 Master Weights:** $4	ext{ bytes}$
- **FP32 First Moment ($m_t$):** $4	ext{ bytes}$
- **FP32 Second Moment ($v_t$):** $4	ext{ bytes}$
- **Total Model + Optimizer State:** $2 + 2 + 4 + 4 + 4 = \mathbf{16	ext{ bytes per parameter}}$ (plus activations!).

Let's compute the complete sizing table for **7B, 13B, and 70B** models and map each configuration to NVIDIA GPUs.

In [5]:
def recommend_nvidia_gpu(vram_gb: float) -> str:
    if vram_gb <= 16:
        return "1x T4 (16GB) / RTX 4080 (16GB)"
    elif vram_gb <= 22:
        return "1x L4 (24GB) / RTX 4090 (24GB)"
    elif vram_gb <= 44:
        return "1x L40S (48GB) / A6000 (48GB)"
    elif vram_gb <= 76:
        return "1x A100 (80GB) / 1x H100 (80GB)"
    elif vram_gb <= 135:
        return "1x H200 (141GB) or 2x A100/H100 (80GB)"
    elif vram_gb <= 300:
        return "4x A100/H100 (80GB)"
    else:
        return "8x H100/H200 Node (640GB - 1.1TB)"


def build_llm_memory_table():
    models = [7, 13, 70]  # Billions of parameters
    precisions = [
        ("FP16 / BF16 (2 B/param)", 2.0),
        ("FP8 / INT8 (1 B/param)", 1.0),
        ("INT4 / NF4 (0.5 B/param)", 0.5),
    ]
    rows = []
    for n_b in models:
        for prec_name, bytes_per_p in precisions:
            weights_gb = n_b * bytes_per_p
            inf_total_gb = round(weights_gb * 1.20, 1)  # +20% KV cache & runtime buffer
            # Full AdamW mixed-precision training vs QLoRA (only ~1.25x on top of quantized base weights)
            adamw_full_train_gb = round(n_b * 16.0 * 1.15, 1) if bytes_per_p == 2.0 else None
            rows.append({
                "Model_Size": f"{n_b}B",
                "Precision": prec_name,
                "Weights_Only_GB": round(weights_gb, 1),
                "Inference_Total_GB (+20%)": inf_total_gb,
                "Full_AdamW_Train_GB": adamw_full_train_gb if adamw_full_train_gb else "N/A (Use LoRA/QLoRA)",
                "Min_Inference_GPU": recommend_nvidia_gpu(inf_total_gb),
            })
    return pd.DataFrame(rows)

df_mem = build_llm_memory_table()
print("=== LLM MEMORY SIZING TABLE (7B, 13B, 70B) ===")
print(df_mem.to_string(index=False))

=== LLM MEMORY SIZING TABLE (7B, 13B, 70B) ===
Model_Size                Precision  Weights_Only_GB  Inference_Total_GB (+20%)  Full_AdamW_Train_GB                      Min_Inference_GPU
        7B  FP16 / BF16 (2 B/param)             14.0                       16.8                128.8         1x L4 (24GB) / RTX 4090 (24GB)
        7B   FP8 / INT8 (1 B/param)              7.0                        8.4 N/A (Use LoRA/QLoRA)         1x T4 (16GB) / RTX 4080 (16GB)
        7B INT4 / NF4 (0.5 B/param)              3.5                        4.2 N/A (Use LoRA/QLoRA)         1x T4 (16GB) / RTX 4080 (16GB)
       13B  FP16 / BF16 (2 B/param)             26.0                       31.2                239.2          1x L40S (48GB) / A6000 (48GB)
       13B   FP8 / INT8 (1 B/param)             13.0                       15.6 N/A (Use LoRA/QLoRA)         1x T4 (16GB) / RTX 4080 (16GB)
       13B INT4 / NF4 (0.5 B/param)              6.5                        7.8 N/A (Use LoRA/QLoRA)         1x T

> **EXAM TIP (Chapter 2):** Memorize the 70B row!
> - **70B at FP16:** $70 	imes 2 = 140	ext{ GB}$ for weights alone ($\sim 168	ext{ GB}$ with KV cache) $
ightarrow$ **Does NOT fit on a single 80GB A100/H100!** Requires at least **2× 80GB GPUs** (with Tensor Parallelism) or **INT4 quantization** ($70 	imes 0.5 = 35	ext{ GB}$ weights, $\sim 42	ext{ GB}$ total, which fits on a single 48GB L40S or 80GB A100/H100).
> - **7B at FP16:** $7 	imes 2 = 14	ext{ GB}$ weights ($\sim 16.8	ext{ GB}$ with KV cache) $
ightarrow$ Fits comfortably on a single **24GB GPU** (L4 / RTX 4090) for inference, or even an **8GB laptop GPU** when quantized to **INT4** ($3.5	ext{ GB}$ weights).

---
## 5. Verifying the Formula on a Real PyTorch Transformer Model

Let's instantiate a real multi-layer Transformer encoder/decoder block in PyTorch and verify that our theoretical formula:
$$	ext{Total Bytes} = \sum_{p \in 	heta} N_p 	imes 	ext{bytes\_per\_element}$$
matches `sum(p.numel() * p.element_size() for p in model.parameters())` down to the exact byte!

In [6]:
class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size: int = 32000, d_model: int = 512, n_heads: int = 8, n_layers: int = 6):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4, batch_first=True
        )
        self.blocks = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        return self.lm_head(self.blocks(self.tok_emb(x)))

model_fp32 = MiniTransformerLM()
total_params = sum(p.numel() for p in model_fp32.parameters())

verification_rows = []
for dtype_label, target_dtype, expected_bytes_per_param in [
    ("FP32 (torch.float32)", torch.float32, 4),
    ("FP16 (torch.float16)", torch.float16, 2),
    ("BF16 (torch.bfloat16)", torch.bfloat16, 2),
]:
    m_cast = MiniTransformerLM().to(dtype=target_dtype)
    formula_bytes = total_params * expected_bytes_per_param
    actual_bytes = sum(p.numel() * p.element_size() for p in m_cast.parameters())
    verification_rows.append({
        "Precision": dtype_label,
        "Total_Parameters": f"{total_params:,}",
        "Formula_MB": round(formula_bytes / (1024 ** 2), 4),
        "Measured_PyTorch_MB": round(actual_bytes / (1024 ** 2), 4),
        "Exact_Byte_Match": formula_bytes == actual_bytes,
    })

df_verify = pd.DataFrame(verification_rows)
print("=== FORMULA VS. MEASURED PYTORCH PARAMETER MEMORY ===")
print(df_verify.to_string(index=False))

=== FORMULA VS. MEASURED PYTORCH PARAMETER MEMORY ===
            Precision Total_Parameters  Formula_MB  Measured_PyTorch_MB  Exact_Byte_Match
 FP32 (torch.float32)       51,682,304    197.1523             197.1523              True
 FP16 (torch.float16)       51,682,304     98.5762              98.5762              True
BF16 (torch.bfloat16)       51,682,304     98.5762              98.5762              True


---
## 6. Hands-On Exercises

### Exercise 1: Exact KV Cache Memory Calculator
Write a function `calculate_kv_cache_gb(n_layers, n_kv_heads, head_dim, seq_len, batch_size, bytes_per_elem=2)` that returns the exact KV cache footprint in GB ($10^9$ bytes).
Use it to calculate the KV cache size for **Llama-3-8B** (`n_layers=32`, `n_kv_heads=8` with Grouped-Query Attention, `head_dim=128`) vs. an equivalent **Multi-Head Attention (MHA)** model (`n_kv_heads=32`) at `seq_len=8192`, `batch_size=16`, in FP16 (`bytes_per_elem=2`). How much memory does **Grouped-Query Attention (GQA)** save?

In [ ]:
# TODO: Complete Exercise 1
def calculate_kv_cache_gb(n_layers: int, n_kv_heads: int, head_dim: int, seq_len: int, batch_size: int, bytes_per_elem: int = 2) -> float:
    pass

### Exercise 2: Can It Fit on My GPU?
A company has a single **NVIDIA L40S GPU (48 GB VRAM)** and wants to serve a **34B parameter coding LLM** with a 20% runtime/KV-cache overhead budget.
Write a short script that checks whether the 34B model fits in 48 GB at **FP16**, **INT8**, and **INT4**, and prints the maximum precision that fits on a single L40S.

In [ ]:
# TODO: Complete Exercise 2

---
## 7. Reference Solutions

In [7]:
# Solution to Exercise 1: KV Cache with GQA vs. Standard MHA
def calculate_kv_cache_gb_solution(
    n_layers: int, n_kv_heads: int, head_dim: int, seq_len: int, batch_size: int, bytes_per_elem: int = 2
) -> float:
    # Factor of 2 for Key + Value tensors
    total_bytes = 2 * n_layers * n_kv_heads * head_dim * seq_len * batch_size * bytes_per_elem
    return total_bytes / 1e9

kv_mha = calculate_kv_cache_gb_solution(n_layers=32, n_kv_heads=32, head_dim=128, seq_len=8192, batch_size=16, bytes_per_elem=2)
kv_gqa = calculate_kv_cache_gb_solution(n_layers=32, n_kv_heads=8, head_dim=128, seq_len=8192, batch_size=16, bytes_per_elem=2)

print("=== Exercise 1: KV Cache Sizing (seq_len=8192, batch_size=16, FP16) ===")
print(f"Standard Multi-Head Attention (32 KV heads) : {kv_mha:.2f} GB")
print(f"Grouped-Query Attention (8 KV heads)        : {kv_gqa:.2f} GB (Saves {kv_mha - kv_gqa:.2f} GB / 4x reduction!)")

# Solution to Exercise 2: 34B Model on 48GB L40S GPU
print("\n=== Exercise 2: 34B Model on 1x NVIDIA L40S (48 GB VRAM) ===")
for name, bpp in [("FP16 (2 B/p)", 2.0), ("INT8 (1 B/p)", 1.0), ("INT4 (0.5 B/p)", 0.5)]:
    req_gb = 34.0 * bpp * 1.20
    fits = req_gb <= 48.0
    print(f"  {name:<15} -> Required VRAM (with 20% buffer): {req_gb:>5.1f} GB | Fits in 48 GB? {fits}")

=== Exercise 1: KV Cache Sizing (seq_len=8192, batch_size=16, FP16) ===
Standard Multi-Head Attention (32 KV heads) : 68.72 GB
Grouped-Query Attention (8 KV heads)        : 17.18 GB (Saves 51.54 GB / 4x reduction!)

=== Exercise 2: 34B Model on 1x NVIDIA L40S (48 GB VRAM) ===
  FP16 (2 B/p)    -> Required VRAM (with 20% buffer):  81.6 GB | Fits in 48 GB? False
  INT8 (1 B/p)    -> Required VRAM (with 20% buffer):  40.8 GB | Fits in 48 GB? True
  INT4 (0.5 B/p)  -> Required VRAM (with 20% buffer):  20.4 GB | Fits in 48 GB? True


---
## 8. NCA-GENL Chapter 2 Review Takeaways

- **GPU Advantage:** Massive SIMT parallelism + specialized **Tensor Cores** + high **HBM memory bandwidth**.
- **BF16 vs. FP16:** Both use 2 bytes/parameter, but **BF16 keeps FP32's 8-bit exponent**, preventing gradient overflow during training.
- **Memory Rule of Thumb:** Weights take $4N$ (FP32), $2N$ (FP16/BF16), $1N$ (INT8/FP8), or $0.5N$ (INT4) GB for an $N$-billion parameter model; add $\sim 20\%$ for KV cache and activations during inference, or $\sim 16	ext{ bytes/param}$ for full mixed-precision AdamW training.